In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import optuna
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from catboost import CatBoostRegressor
from xgboost import XGBRegressor
from sklearn.model_selection import cross_val_score
from sklearn.metrics import root_mean_squared_error

In [8]:
data = pd.read_csv("../datasets/Exam_Score_Prediction.csv")

X = data.drop('exam_score', axis=1)
y = data['exam_score']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=True, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.1, shuffle=True, random_state=42)

In [9]:
class AddStudySleepRatio(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        X = X.copy()
        X["study_sleep_ratio"] = X["study_hours"] / X["sleep_hours"]
        return X

preprocessing_pipeline = joblib.load("../models_and_pipelines/preprocessing_pipeline.pkl")

In [10]:
X_train = preprocessing_pipeline.transform(X_train)
X_val = preprocessing_pipeline.transform(X_val)
X_test = preprocessing_pipeline.transform(X_test)

In [ ]:
def objective(trial, model_name):
    if model_name == "Linear Regression": params = {}

    elif model_name == "Lasso": params = {
        "alpha": trial.suggest_float("alpha", 0.1, 10, step=0.1),
        "tol": trial.suggest_float("tol", 0.001, 1, step=0.001)
    }

    elif model_name == "Ridge": params = {
        "alpha": trial.suggest_float("alpha", 0.1, 10, step=0.1),
        "tol": trial.suggest_float("tol", 0.001, 1, step=0.001)
    }

    elif model_name == "K-Nearest Neighbors": params = {
        "n_neighbors": trial.suggest_int("n_neighbors", 1, 100),
        "leaf_size": trial.suggest_int("leaf_size", 10, 1000, step=10)
    }

    elif model_name == "Support Vector Regression": params = {
        "degree": trial.suggest_int("degree", 1, 10),
        "tol": trial.suggest_float("tol", 0.001, 1, step=0.001),
        "C": trial.suggest_float("C", 1, 1000, step=0.5),
        "epsilon": trial.suggest_float("epsilon", 0.1, 100, step=0.1),
    }

    elif model_name == "Decision Tree Regression": params = {
        "max_depth": trial.suggest_int("max_depth", 1, 100),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 2, 1000),
    }

    elif model_name == "Random Forest Regression": params = {
        "n_estimators": trial.suggest_int("n_estimators", 100, 500, 10),
        "max_depth": trial.suggest_int("max_depth", 1, 100),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 2, 1000),
    }

    elif model_name == "CatBoosting Regression": params = {
        "iterations": trial.suggest_int("iterations", 1, 501, step=10),
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.5, step=0.05),
        "depth": trial.suggest_int("depth", 1, 10),
    }

    elif model_name == "XGBusting Regression": params = {
        "learning_rate": trial.suggest_float("learning_rate", 0.05, 0.5, step=0.05),
        "max_depth": trial.suggest_int("max_depth", 1, 100),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
        "gamma": trial.suggest_float("gamma", 0, 1, step=0.1),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0, 1, step=0.1),
    }

    model = models[model_name]
    model.set_params(**params)

    score = cross_val_score(model, X_train, y_train, cv=3, n_jobs=-1, scoring="neg_root_mean_squared_error")
    final_score = abs(score.mean())

    return final_score
    
models = {
        "Linear Regression": LinearRegression(n_jobs=-1),
        "Lasso": Lasso(), 
        "Ridge": Ridge(),
        "K-Nearest Neighbors": KNeighborsRegressor(n_jobs=-1),
        "Support Vector Regression": SVR(max_iter=10000),
        "Decision Tree Regression": DecisionTreeRegressor(),
        "Random Forest Regression": RandomForestRegressor(n_jobs=-1),
        "CatBoosting Regression": CatBoostRegressor(),
        "XGBusting Regression": XGBRegressor()
        }

models_test_score = {}
models_best_params = {}

for model_name in models.keys():
    study = optuna.create_study(direction="minimize")
    study.optimize(lambda trial: objective(trial, model_name), n_trials=100)

    models_test_score[model_name] = study.best_trial.values[0]
    models_best_params[model_name] = study.best_params

[I 2026-02-22 12:22:13,813] A new study created in memory with name: no-name-80cb3a70-b707-42cf-be28-0db74900dc70
[I 2026-02-22 12:22:13,947] Trial 0 finished with value: 9.802460898402996 and parameters: {}. Best is trial 0 with value: 9.802460898402996.
[I 2026-02-22 12:22:14,083] Trial 1 finished with value: 9.802460898402996 and parameters: {}. Best is trial 0 with value: 9.802460898402996.
[I 2026-02-22 12:22:14,218] Trial 2 finished with value: 9.802460898402996 and parameters: {}. Best is trial 0 with value: 9.802460898402996.
[I 2026-02-22 12:22:14,352] Trial 3 finished with value: 9.802460898402996 and parameters: {}. Best is trial 0 with value: 9.802460898402996.
[I 2026-02-22 12:22:14,487] Trial 4 finished with value: 9.802460898402996 and parameters: {}. Best is trial 0 with value: 9.802460898402996.
[I 2026-02-22 12:22:14,620] Trial 5 finished with value: 9.802460898402996 and parameters: {}. Best is trial 0 with value: 9.802460898402996.
[I 2026-02-22 12:22:14,756] Trial 

In [13]:
models_val_score = {}
best_models = {}

for model_name in models.keys():
    model = models[model_name]
    params = models_best_params[model_name]
    model.set_params(**params)
    model.fit(X_test, y_test)

    y_val_pred = model.predict(X_val)
    models_val_score[model_name] = root_mean_squared_error(y_val, y_val_pred)
    best_models[model_name] = model

c:\python\Exam-Score-Prediction-Project\venv\Lib\site-packages\sklearn\base.py:1336: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
c:\python\Exam-Score-Prediction-Project\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
c:\python\Exam-Score-Prediction-Project\venv\Lib\site-packages\sklearn\linear_model\_coordinate_descent.py:716: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 1.900e+05, tolerance: 7.440e+04
Linear regression models with a zero l1 penalization strength are more efficiently fitted using one of the solvers implemented in sklearn.linear_model.Ridge/

0:	learn: 18.4511472	total: 150ms	remaining: 33.1s
1:	learn: 17.9853441	total: 154ms	remaining: 16.8s
2:	learn: 17.5469461	total: 157ms	remaining: 11.4s
3:	learn: 17.1165758	total: 161ms	remaining: 8.71s
4:	learn: 16.7072568	total: 163ms	remaining: 7.03s
5:	learn: 16.3330577	total: 165ms	remaining: 5.91s
6:	learn: 15.9642634	total: 167ms	remaining: 5.09s
7:	learn: 15.6329641	total: 168ms	remaining: 4.48s
8:	learn: 15.3319529	total: 170ms	remaining: 4s
9:	learn: 15.0293156	total: 171ms	remaining: 3.6s
10:	learn: 14.7860766	total: 172ms	remaining: 3.28s
11:	learn: 14.5143951	total: 173ms	remaining: 3.01s
12:	learn: 14.2800590	total: 174ms	remaining: 2.78s
13:	learn: 14.0625363	total: 175ms	remaining: 2.58s
14:	learn: 13.8444993	total: 175ms	remaining: 2.41s
15:	learn: 13.6545448	total: 176ms	remaining: 2.25s
16:	learn: 13.4915335	total: 177ms	remaining: 2.12s
17:	learn: 13.2940072	total: 177ms	remaining: 2s
18:	learn: 13.1250480	total: 178ms	remaining: 1.89s
19:	learn: 12.9697639	total: 

In [14]:
final_model_name = min(models_val_score, key=models_val_score.get)
final_model_val_score = models_val_score[final_model_name]
final_model = best_models[final_model_name]

In [15]:
final_model_name, final_model_val_score

('Ridge', 9.843366916846959)

In [16]:
y_test_pred = final_model.predict(X_test)
final_model_test_score = root_mean_squared_error(y_test, y_test_pred)

In [17]:
final_model_test_score

9.7457453535136

In [18]:
joblib.dump(final_model, "../models_and_pipelines/final_model.pkl")

['../models_and_pipelines/final_model.pkl']